In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T


class Inception(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        ch1x1,
        ch3x3pre,
        ch3x3,
        ch5x5pre,
        ch5x5,
        pool_proj,
    ):
        super(Inception, self).__init__()

        # 1x1 卷积分支
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, kernel_size=1), nn.ReLU(inplace=True)
        )

        # 3x3 卷积分支
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3pre, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch3x3pre, ch3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

        # 5x5 卷积分支
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5pre, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch5x5pre, ch5x5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
        )

        # 池化分支
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


class InceptionAux(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super(InceptionAux, self).__init__()
        self.features = nn.Sequential(
            nn.AvgPool2d(kernel_size=5, stride=3),
            nn.Conv2d(in_channels, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Dropout(0.7),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.7),
            nn.Linear(128 * 4 * 4, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# 输入128*128
class GoogleNet(nn.Module):
    def __init__(self, num_classes=1000, aux_logits=True, init_weights=True):
        super(GoogleNet, self).__init__()
        self.aux_logits = aux_logits
        self.init_weights = init_weights
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.inception3a = Inception(192, 256, 64, 96, 128, 16, 32, 32)
        self.inception3b = Inception(256, 480, 128, 128, 192, 32, 96, 64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.inception4a = Inception(480, 512, 192, 96, 208, 16, 48, 64)
        self.inception4b = Inception(512, 512, 160, 112, 224, 24, 64, 64)
        self.inception4c = Inception(512, 512, 128, 128, 256, 24, 64, 64)
        self.inception4d = Inception(512, 528, 112, 144, 288, 32, 64, 64)
        self.inception4e = Inception(528, 832, 256, 160, 320, 32, 128, 128)
        self.inception5a = Inception(832, 832, 256, 160, 320, 32, 128, 128)
        self.inception5b = Inception(832, 1024, 384, 192, 384, 48, 128, 128)
        self.avgpool = nn.AvgPool2d(kernel_size=7, stride=1)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, num_classes)

        if self.training and aux_logits:
            self.aux1 = InceptionAux(num_classes, in_channels=512)
            self.aux2 = InceptionAux(num_classes, in_channels=528)

        if self.init_weights:
            self.initialize_weights()

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool(x)
        x = self.inception4a(x)
        if self.training and self.aux_logits:
            aux1 = self.aux1(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        if self.training and self.aux_logits:
            aux2 = self.aux2(x)
        x = self.inception4e(x)
        x = self.inception5a(x)
        x = self.inception5b(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        if self.training and self.aux_logits:
            return x, aux1, aux2
        return x

    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import matplotlib.pyplot as plt


class DataSetLoader_test(Dataset):
    def __init__(self, root_dir, transform=None, num_classes=101, return_pil=False):
        self.root_dir = root_dir
        self.transform = transform
        self.num_classes = num_classes
        # 如果 return_pil 为 True，则 __getitem__ 返回 PIL 图像而不是张量
        self.return_pil = return_pil

        # 收集所有 (img_path, label_idx)
        self.samples = []
        self.class_names = sorted(
            [
                d
                for d in os.listdir(root_dir)
                if os.path.isdir(os.path.join(root_dir, d))
            ]
        )[: self.num_classes]
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.class_names)}

        for cls_name in self.class_names:
            cls_dir = os.path.join(root_dir, cls_name)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    img_path = os.path.join(cls_dir, fname)
                    label = self.class_to_idx[cls_name]
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.return_pil:
            return image, label

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
import random


class DataSetLoader_train(Dataset):
    def __init__(
        self,
        root_dir,
        transform=None,
        num_classes=101,
        split_ratio=0.8,
        return_pil=False,
        dataclass="train",
    ):
        self.root_dir = root_dir
        self.transform = transform
        self.num_classes = num_classes
        self.split_ratio = split_ratio
        self.return_pil = return_pil
        self.dataclass = dataclass
        self.trainSamples = []
        self.validSamples = []
        self.class_names = sorted(
            [
                d
                for d in os.listdir(root_dir)
                if os.path.isdir(os.path.join(root_dir, d))
            ]
        )[: self.num_classes]
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.class_names)}
        self.train_count = {cls_name: 0 for cls_name in self.class_names}
        self.valid_count = {cls_name: 0 for cls_name in self.class_names}

        for cls_name in self.class_names:
            cls_dir = os.path.join(root_dir, cls_name)
            cls_samples = []
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    img_path = os.path.join(cls_dir, fname)
                    label = self.class_to_idx[cls_name]
                    cls_samples.append((img_path, label))
            random.shuffle(cls_samples)
            split_idx = int(len(cls_samples) * self.split_ratio)
            train_split = cls_samples[:split_idx]
            valid_split = cls_samples[split_idx:]
            self.trainSamples.extend(train_split)  # 前80% 训练样本
            self.validSamples.extend(valid_split)  # 后20% 验证样本
            self.train_count[cls_name] += len(train_split)
            self.valid_count[cls_name] += len(valid_split)

    def __len__(self):
        if self.dataclass == "train":
            return len(self.trainSamples)
        else:
            return len(self.validSamples)

    def __getitem__(self, idx):
        if self.dataclass == "train":
            img_path, label = self.trainSamples[idx]
        else:
            img_path, label = self.validSamples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.return_pil:
            return image, label

        if self.transform is not None:
            image = self.transform(image)
        else:
            transform = T.Compose(
                [
                    T.Resize((128, 128)),
                    T.ToTensor(),
                    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ]
            )
            image = transform(image)

        return image, label

In [ ]:
from tqdm import tqdm


def test_val(model, val_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="验证中"):
            images, labels = images.to(device), labels.to(device)
            if images.dim() == 5:
                bsz, ncrops, c, h, w = images.size()
                outputs = model(images.view(-1, c, h, w))
                outputs = outputs.view(bsz, ncrops, -1).mean(dim=1)
            else:
                outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / max(total, 1)
    accuracy = correct / max(total, 1)
    return accuracy


def train(
    model,
    epoch,
    train_loader,
    optimizer,
    criterion,
    device,
    scheduler=None,
    val_loader=None,
):
    train_acc = []
    val_acc = []
    loss_history = []

    for ep in range(epoch):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"训练中 [{ep+1}/{epoch}]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            # 支持辅助分类器损失加权
            if isinstance(outputs, tuple) and len(outputs) == 3:
                main_out, aux1_out, aux2_out = outputs
                main_loss = criterion(main_out, labels)
                aux1_loss = criterion(aux1_out, labels)
                aux2_loss = criterion(aux2_out, labels)
                batch_loss = main_loss + 0.3 * aux1_loss + 0.3 * aux2_loss
            else:
                batch_loss = criterion(outputs, labels)
            batch_loss.backward()
            optimizer.step()

            total_loss += batch_loss.item() * labels.size(0)
            if isinstance(outputs, tuple):
                _, predicted = main_out.max(1)
            else:
                _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        avg_loss = total_loss / max(total, 1)
        accuracy = correct / max(total, 1)
        train_acc.append(accuracy)
        loss_history.append(avg_loss)

        val_acc_ = None
        if val_loader is not None:
            val_acc_ = test_val(model, val_loader, criterion, device)
            val_acc.append(val_acc_ * 0.5 + accuracy * 0.5)

        if scheduler is not None:
            if val_acc_ is None:
                scheduler.step()
            else:
                scheduler.step(val_acc_)

        if val_acc_ is None:
            print(
                f"Epoch [{ep+1}/{epoch}] 平均损失: {avg_loss:.4f} 训练准确率: {accuracy:.4f}"
            )
        else:
            print(
                f"Epoch [{ep+1}/{epoch}] 平均损失: {avg_loss:.4f} 训练准确率: {accuracy:.4f} 验证准确率: {val_acc_:.4f}"
            )

    return train_acc, loss_history, val_acc

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_acc_loss(train_acc, loss, val_acc=None):
    epochs = np.arange(1, len(train_acc) + 1)
    plt.figure(figsize=(16, 5))

    # --- 准确率曲线 ---
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_acc, "o-", label="Train Acc", color="C0")
    if val_acc is not None and len(val_acc) == len(train_acc):
        plt.plot(epochs, val_acc, "o-", label="Val Acc", color="C1")
        best_val_idx = np.argmax(val_acc)
        plt.scatter(epochs[best_val_idx], val_acc[best_val_idx], color="red", label="Best Val Acc")
        plt.text(epochs[best_val_idx], val_acc[best_val_idx], f"{val_acc[best_val_idx]:.4f}", color="red", fontsize=12, va="bottom")
    best_train_idx = np.argmax(train_acc)
    plt.scatter(epochs[best_train_idx], train_acc[best_train_idx], color="blue", label="Best Train Acc")
    plt.text(epochs[best_train_idx], train_acc[best_train_idx], f"{train_acc[best_train_idx]:.4f}", color="blue", fontsize=12, va="bottom")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curve")
    plt.legend()
    plt.grid(True)

    # --- 损失曲线 ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, "o-", label="Train Loss", color="orange")
    best_loss_idx = np.argmin(loss)
    plt.scatter(epochs[best_loss_idx], loss[best_loss_idx], color="green", label="Best Train Loss")
    plt.text(epochs[best_loss_idx], loss[best_loss_idx], f"{loss[best_loss_idx]:.5f}", color="green", fontsize=12, va="bottom")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
import os
from datetime import datetime

# 保存/加载检查点的实用函数（在新的 notebook cell 中执行）
def save_checkpoint(path, model, optimizer, scheduler=None, epoch=None, best_acc=None):
  os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
  # 如果使用了 DataParallel，保存 module 的 state_dict
  state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
  checkpoint = {
    "epoch": epoch,
    "model_state_dict": state_dict,
    "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
    "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
    "best_acc": best_acc,
    "saved_at": datetime.utcnow().isoformat(),
  }
  torch.save(checkpoint, path)
  print(f"Saved checkpoint to {path}")

In [ ]:
def test(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="测试中"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    print(f"测试准确率: {accuracy:.4f}")

In [ ]:
train_dir = "/data/data_taohy/awesomeCopression/food101_images/train"
test_dir = "/data/data_taohy/awesomeCopression/food101_images/validation"

NUM_CLASSES = 5
BATCH_SIZE = 64
mean = [0.5449, 0.4433, 0.3435]
std = [0.2271, 0.2376, 0.2350]

transform_train = T.Compose(
    [
        T.RandomResizedCrop(128),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ]
)

transform_val = T.Compose(
    [
        T.Resize((128, 128)),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ]
)


train_dataset = DataSetLoader_train(
    root_dir=train_dir,
    num_classes=NUM_CLASSES,
    transform=transform_train,
    dataclass="train",
    return_pil=False,
)

val_dataset = DataSetLoader_train(
    root_dir=train_dir,
    num_classes=NUM_CLASSES,
    transform=transform_val,
    dataclass="valid",
    return_pil=False,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8
)

transform_test = T.Compose(
    [
        T.Resize((128, 128)),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ]
)

test_loader = DataLoader(
    DataSetLoader_test(
        root_dir=test_dir,
        num_classes=NUM_CLASSES,
        transform=transform_test,
        return_pil=False,
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GoogleNet(num_classes=NUM_CLASSES, aux_logits=True)
if torch.cuda.device_count() > 1:
    print("使用多卡训练, GPU 数量:", torch.cuda.device_count())

    model = nn.DataParallel(model, device_ids=[0, 1, 2, 3])
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4
)

使用多卡训练, GPU 数量: 4


In [ ]:
train_acc, loss_history, val_acc = train(
    model,
    epoch=30,
    train_loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scheduler=scheduler,
    val_loader=val_loader,  
)

训练中 [1/30]:   0%|          | 0/47 [00:02<?, ?it/s]


AttributeError: Caught AttributeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/data/xyjiang/anaconda3/envs/thy_env1/lib/python3.11/site-packages/torch/nn/parallel/parallel_apply.py", line 96, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/xyjiang/anaconda3/envs/thy_env1/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/xyjiang/anaconda3/envs/thy_env1/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1750, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/xyjiang/tmp/ipykernel_3692112/3578412849.py", line 126, in forward
    if self.training and self.aux_logits:
                         ^^^^^^^^^^^^^^^
  File "/data/xyjiang/anaconda3/envs/thy_env1/lib/python3.11/site-packages/torch/nn

In [ ]:
plot_acc_loss(train_acc, loss_history,val_acc)
save_checkpoint("checkpoints/googlenet_food101-30.pth", model, optimizer, scheduler, epoch=30, best_acc=max(val_acc))

In [ ]:
test(test_loader=test_loader, model=model, device=device)

In [ ]:
train_acc1, loss_history1, val_acc1 = train(
    model,
    epoch=50,
    train_loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scheduler=scheduler,
    val_loader=val_loader,  
)

In [ ]:
plot_acc_loss(train_acc + train_acc1, loss_history + loss_history1, val_acc + val_acc1)
save_checkpoint("checkpoints/googlenet_food101-80.pth", model, optimizer, scheduler, epoch=80, best_acc=max(val_acc + val_acc1))

In [ ]:
test(test_loader=test_loader, model=model, device=device)